### Task 1: Extract Data from JSONPlaceholder API

I will extract data from the `/users` and `/posts` endpoints of the JSONPlaceholder API. To ensure fault tolerance, I'll implement a retry mechanism with exponential backoff and handle various HTTP error codes and timeouts.

After extraction, I will normalize the nested JSON fields into a flat DataFrame using `pd.json_normalize()`.

In [12]:
import requests
import pandas as pd
import time

# Base URL for JSONPlaceholder API
BASE_URL = 'https://jsonplaceholder.typicode.com'

def fetch_data_with_retries(endpoint, max_retries=5, backoff_factor=0.5):
    """Fetches data from a given API endpoint with retries and exponential backoff."""
    for i in range(max_retries):
        try:
            response = requests.get(f'{BASE_URL}/{endpoint}', timeout=10) # 10-second timeout
            response.raise_for_status()  # Raise an HTTPError for bad responses (4xx or 5xx)
            return response.json()
        except requests.exceptions.HTTPError as e:
            print(f"HTTP error fetching {endpoint}: {e}")
            if 400 <= response.status_code < 500 and response.status_code != 429: # Client error, not retryable (except Too Many Requests)
                print(f"Skipping retry for client error {response.status_code}")
                break
        except requests.exceptions.Timeout:
            print(f"Timeout fetching {endpoint}. Retrying...")
        except requests.exceptions.ConnectionError as e:
            print(f"Connection error fetching {endpoint}: {e}. Retrying...")
        except requests.exceptions.RequestException as e:
            print(f"An unexpected error occurred fetching {endpoint}: {e}. Retrying...")

        sleep_time = backoff_factor * (2 ** i)
        print(f"Retrying {endpoint} in {sleep_time:.2f} seconds...")
        time.sleep(sleep_time)
    print(f"Failed to fetch {endpoint} after {max_retries} retries.")
    return None

# Fetch users and posts data
users_data = fetch_data_with_retries('users')
posts_data = fetch_data_with_retries('posts')

# Normalize users data
if users_data:
    df_users = pd.json_normalize(users_data)
    print("Users DataFrame Head:")
    display(df_users.head())
else:
    df_users = pd.DataFrame()
    print("Failed to load users data.")

# Normalize posts data
if posts_data:
    df_posts = pd.json_normalize(posts_data)
    print("\nPosts DataFrame Head:")
    display(df_posts.head())
else:
    df_posts = pd.DataFrame()
    print("Failed to load posts data.")


Users DataFrame Head:


,id,name,username,email,phone,website,address.street,address.suite,address.city,address.zipcode,address.geo.lat,address.geo.lng,company.name,company.catchPhrase,company.bs
0,1,Leanne Graham,Bret,Sincere@april.biz,1-770-736-8031 x56442,hildegard.org,Kulas Light,Apt. 556,Gwenborough,92998-3874,-37.3159,81.1496,Romaguera-Crona,Multi-layered client-server neural-net,harness real-time e-markets
1,2,Ervin Howell,Antonette,Shanna@melissa.tv,010-692-6593 x09125,anastasia.net,Victor Plains,Suite 879,Wisokyburgh,90566-7771,-43.9509,-34.4618,Deckow-Crist,Proactive didactic contingency,synergize scalable supply-chains
2,3,Clementine Bauch,Samantha,Nathan@yesenia.net,1-463-123-4447,ramiro.info,Douglas Extension,Suite 847,McKenziehaven,59590-4157,-68.6102,-47.0653,Romaguera-Jacobson,Face to face bifurcated interface,e-enable strategic applications
3,4,Patricia Lebsack,Karianne,Julianne.OConner@kory.org,493-170-9623 x156,kale.biz,Hoeger Mall,Apt. 692,South Elvis,53919-4257,29.4572,-164.2990,Robel-Corkery,Multi-tiered zero tolerance productivity,transition cutting-edge web services
4,5,Chelsey Dietrich,Kamren,Lucio_Hettinger@annie.ca,(254)954-1289,demarco.info,Skiles Walks,Suite 351,Roscoeview,33263,-31.8129,62.5342,Keebler LLC,User-centric fault-tolerant solution,revolutionize end-to-end systems



Posts DataFrame Head:


,userId,id,title,body
0,1,1,sunt aut facere repellat provident occaecati e...,quia et suscipit\nsuscipit recusandae consequu...
1,1,2,qui est esse,est rerum tempore vitae\nsequi sint nihil repr...
2,1,3,ea molestias quasi exercitationem repellat qui...,et iusto sed quo iure\nvoluptatem occaecati om...
3,1,4,eum et est occaecati,ullam et saepe reiciendis voluptatem adipisci\...
4,1,5,nesciunt quas odio,repudiandae veniam quaerat sunt sed\nalias aut...


Now that we have extracted and normalized the data from the API, the next step is to generate the messy CSV file locally and then extract data from it. After that, we can proceed with merging and cleaning.

### Task 1 (Continued): Generate and Extract Data from a Messy CSV File

I will now create a messy CSV file locally to simulate the second data source. This file will contain some intentional inconsistencies, missing values, and potential duplicates to be addressed later during the cleaning phase. After generating the file, I will load it into a pandas DataFrame.

In [13]:
import pandas as pd
import numpy as np

# Define messy data for the CSV file
messy_csv_data = {
    'user_id': [1, 2, 3, 4, 5, 1, 6, 7, 8, 9],
    'name': ['John Doe', 'Jane Smith', 'JOHN DOE', 'Alice Brown', 'Bob White', 'John Doe', 'Charlie Green', 'David Blue', 'Eva Black', 'Frank Yellow'],
    'email': ['john.doe@example.com', 'jane.smith@example.com', 'john.doe@example.com', 'alice.brown@example.com', 'bob.white@example.com', 'JOHN.DOE@EXAMPLE.COM', 'charlie.green@example.com', 'david.blue@example.com', 'eva.black@example.com', 'frank.yellow@example.com'],
    'age': [30, 24, 30, np.nan, 45, 30, 29, 31, 25, 50],
    'city': ['New York ', 'Los Angeles', 'new york', 'Chicago', 'Houston', 'New York', 'Seattle', 'Boston', 'Miami', 'Dallas'],
    'registration_date': ['2023-01-15', '2023-02-20', '2023-01-15', '2022-11-01', '2023-03-10', '2023-01-15', '2023-04-05', '2023-05-12', '2023-06-18', '2023-07-22'],
    'occupation': ['Engineer', 'Designer', 'Engineer', 'Doctor', 'Artist', 'Engineer', 'Engineer', 'Lawyer', 'Analyst', 'Manager']
}

df_messy_csv = pd.DataFrame(messy_csv_data)

# Introduce some additional messiness:
# Add a row with extra whitespace and inconsistent casing
df_messy_csv.loc[len(df_messy_csv)] = [10, ' Sarah Connor ', ' sarah.connor@example.com ', 35, ' San Francisco ', '2023-08-01', 'Developer']

# Save to a CSV file
csv_file_path = 'messy_local_data.csv'
df_messy_csv.to_csv(csv_file_path, index=False)

print(f"Messy CSV file '{csv_file_path}' created successfully.")

# Load the messy CSV data into a DataFrame
df_local_csv = pd.read_csv(csv_file_path)

print("\nLocal CSV DataFrame Head:")
display(df_local_csv.head())

print("\nLocal CSV DataFrame Info:")
df_local_csv.info()


Messy CSV file 'messy_local_data.csv' created successfully.

Local CSV DataFrame Head:


,user_id,name,email,age,city,registration_date,occupation
0,1,John Doe,john.doe@example.com,30.0,New York,2023-01-15,Engineer
1,2,Jane Smith,jane.smith@example.com,24.0,Los Angeles,2023-02-20,Designer
2,3,JOHN DOE,john.doe@example.com,30.0,new york,2023-01-15,Engineer
3,4,Alice Brown,alice.brown@example.com,NaN,Chicago,2022-11-01,Doctor
4,5,Bob White,bob.white@example.com,45.0,Houston,2023-03-10,Artist



Local CSV DataFrame Info:
<class 'pandas.DataFrame'>
RangeIndex: 11 entries, 0 to 10
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   user_id            11 non-null     int64  
 1   name               11 non-null     str    
 2   email              11 non-null     str    
 3   age                10 non-null     float64
 4   city               11 non-null     str    
 5   registration_date  11 non-null     str    
 6   occupation         11 non-null     str    
dtypes: float64(1), int64(1), str(5)
memory usage: 748.0 bytes


Now that we have both the API data (`df_users`, `df_posts`) and the local CSV data (`df_local_csv`), the next crucial step is to merge these data sources. Given the task requirements, a key part of this will be handling conflicts, specifically when the same email appears in both sources with different name values. We need to define a conflict resolution strategy.

### Task 1 (Continued): Merge DataFrames and Resolve Conflicts

Now, I will merge `df_users` (from JSONPlaceholder API) and `df_local_csv` on a shared key. Before merging, I'll standardize the `email` column in both DataFrames to ensure consistent matching.

**Conflict Resolution Strategy:**
If the same email appears in both sources with different name values, the `name` from `df_users` (API source) will take precedence. This decision is based on the assumption that the API provides more up-to-date or primary user information.

In [ ]:
# Prepare df_users for merging
# Select relevant columns and rename for consistency with df_local_csv where applicable
df_users_cleaned = df_users[['id', 'name', 'email', 'phone', 'website', 'address.city', 'company.name']].copy()
df_users_cleaned.rename(columns={'id': 'user_id', 'address.city': 'city', 'company.name': 'company'}, inplace=True)


df_users_cleaned['email'] = df_users_cleaned['email'].str.lower().str.strip()
df_local_csv['email'] = df_local_csv['email'].str.lower().str.strip()

# Perform an outer merge to keep all records from both DataFrames
# We'll use suffixes to identify original columns in case of conflicts
merged_df = pd.merge(
    df_local_csv,
    df_users_cleaned,
    on='email',
    how='outer',
    suffixes=('_local', '_api')
)

# Conflict Resolution for 'name' and other fields where API takes precedence
# Create a new 'name' column based on the API data if available, otherwise local
merged_df['name'] = merged_df['name_api'].fillna(merged_df['name_local'])

# For other fields, prioritize API data if available
merged_df['user_id'] = merged_df['user_id_api'].fillna(merged_df['user_id_local'])
merged_df['city'] = merged_df['city_api'].fillna(merged_df['city_local'])

# Drop the original local and API specific columns for name, user_id, and city
merged_df.drop(columns=[
    'name_local', 'name_api',
    'user_id_local', 'user_id_api',
    'city_local', 'city_api'
], inplace=True)

# Reorder columns to have 'user_id', 'name', 'email' at the beginning
# and then other columns, including new ones from API
ordered_columns = [
    'user_id', 'name', 'email', 'age', 'registration_date', 'occupation',
    'phone', 'website', 'company'
]

# Filter to keep only columns that exist in the merged_df to avoid errors
final_columns = [col for col in ordered_columns if col in merged_df.columns]

# Add any remaining columns that were not in ordered_columns
remaining_columns = [col for col in merged_df.columns if col not in final_columns]
final_columns.extend(remaining_columns)

merged_df = merged_df[final_columns]

print("Merged DataFrame with Conflict Resolution Head:")
display(merged_df.head())

print("\nMerged DataFrame Info:")
merged_df.info()

Merged DataFrame with Conflict Resolution Head:


,user_id,name,email,age,registration_date,occupation,phone,website,company,city
0,4.0,Alice Brown,alice.brown@example.com,NaN,2022-11-01,Doctor,NaN,NaN,NaN,Chicago
1,5.0,Bob White,bob.white@example.com,45.0,2023-03-10,Artist,NaN,NaN,NaN,Houston
2,9.0,Glenna Reichert,chaim_mcdermott@dana.io,NaN,NaN,NaN,(775)976-6794 x41206,conrad.com,Yost and Sons,Bartholomebury
3,6.0,Charlie Green,charlie.green@example.com,29.0,2023-04-05,Engineer,NaN,NaN,NaN,Seattle
4,7.0,David Blue,david.blue@example.com,31.0,2023-05-12,Lawyer,NaN,NaN,NaN,Boston



Merged DataFrame Info:
<class 'pandas.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   user_id            21 non-null     float64
 1   name               21 non-null     str    
 2   email              21 non-null     str    
 3   age                10 non-null     float64
 4   registration_date  11 non-null     str    
 5   occupation         11 non-null     str    
 6   phone              10 non-null     str    
 7   website            10 non-null     str    
 8   company            10 non-null     str    
 9   city               21 non-null     str    
dtypes: float64(2), str(8)
memory usage: 1.8 KB


The DataFrames are now merged, and conflicts for `name` and other user-identifying fields have been resolved, prioritizing the API data. The next phase is to apply the 6 cleaning techniques (nulls, duplicates, casing, types, whitespace, outliers) to this `merged_df`.

### Task 1 (Continued): Apply Data Cleaning Techniques

Now, I will apply the 6 specified data cleaning techniques to the `merged_df`:
1.  **Handle Null Values**: Address missing data, potentially by imputation or removal.
2.  **Remove Duplicates**: Identify and remove redundant rows.
3.  **Standardize Casing**: Ensure consistent casing for text fields (e.g., lowercase).
4.  **Correct Data Types**: Convert columns to their appropriate data types (e.g., `age` to integer, `registration_date` to datetime).
5.  **Trim Whitespace**: Remove leading/trailing whitespace from string columns.
6.  **Identify Outliers**: Detect and decide on a strategy for handling outliers, particularly in numerical columns like `age`.

In [15]:
import numpy as np

# Make a copy to preserve the original merged_df if needed
cleaned_df = merged_df.copy()

print("--- Starting Data Cleaning ---")

# 1. Handle Null Values
# For 'age': Impute with median, as it's numerical and contains NaNs.
# For 'registration_date': Fill NaNs with a placeholder or drop rows, depending on requirements.
# For other object columns (phone, website, company, occupation), fill with 'Unknown' or an empty string.

# Age imputation
median_age = cleaned_df['age'].median()
cleaned_df['age'] = cleaned_df['age'].fillna(median_age, inplace=True)
print(f"Filled 'age' nulls with median: {median_age}")

# Registration date - for simplicity, we'll fill with a common date or the mode.
# Convert to datetime first to handle properly
cleaned_df['registration_date'] = pd.to_datetime(cleaned_df['registration_date'], errors='coerce')
mode_reg_date = cleaned_df['registration_date'].mode()[0]
cleaned_df['registration_date'] = cleaned_df['registration_date'].fillna(mode_reg_date, inplace=True)
print(f"Filled 'registration_date' nulls with mode: {mode_reg_date}")

# Fill nulls in other string columns with 'Unknown' or empty string
string_cols_to_fill = ['occupation', 'phone', 'website', 'company']
for col in string_cols_to_fill:
    if col in cleaned_df.columns:
        cleaned_df[col] = cleaned_df[col].fillna('Unknown', inplace=True)
print("Filled nulls in string columns with 'Unknown'.")

# 2. Remove Duplicates
# Considering 'email' as a primary identifier for duplicates
before_dedup = len(cleaned_df)
cleaned_df.drop_duplicates(subset=['email'], keep='first', inplace=True)
after_dedup = len(cleaned_df)
print(f"Removed {before_dedup - after_dedup} duplicate rows based on 'email'.")

# 3. Standardize Casing
# Convert all relevant string columns to lowercase or title case
string_cols = cleaned_df.select_dtypes(include='object').columns
for col in string_cols:
    if col in ['name', 'city', 'occupation', 'company']:
        cleaned_df[col] = cleaned_df[col].str.title() # Title case for proper names
    elif col in ['email', 'website']:
        cleaned_df[col] = cleaned_df[col].str.lower() # Lowercase for email and website
print("Standardized casing for string columns.")

# 4. Correct Data Types
# 'user_id' to int, 'age' to int, 'registration_date' to datetime
cleaned_df['user_id'] = cleaned_df['user_id'].astype(int)
cleaned_df['age'] = cleaned_df['age'].fillna(0).astype(int)
# 'registration_date' was already converted to datetime during null handling
print("Corrected data types for 'user_id', 'age', and 'registration_date'.")

# 5. Trim Whitespace
# Apply to all string columns
for col in string_cols:
    cleaned_df[col] = cleaned_df[col].apply(lambda x: x.strip() if isinstance(x, str) else x)
print("Trimmed whitespace from string columns.")

# 6. Identify Outliers (Example: Age)
# Using IQR method for outlier detection in 'age'
Q1 = cleaned_df['age'].quantile(0.25)
Q3 = cleaned_df['age'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers_age = cleaned_df[(cleaned_df['age'] < lower_bound) | (cleaned_df['age'] > upper_bound)]

if not outliers_age.empty:
    print(f"\nIdentified {len(outliers_age)} potential outliers in 'age' column. Reviewing...")
    display(outliers_age[['name', 'age']])
    # For simplicity, we'll keep them for now, but in a real scenario, we might cap or remove them.
else:
    print("\nNo significant outliers detected in 'age' column based on IQR method.")

print("\n--- Data Cleaning Complete ---")
print("\nCleaned DataFrame Head:")
display(cleaned_df.head())

print("\nCleaned DataFrame Info:")
cleaned_df.info()


--- Starting Data Cleaning ---
Filled 'age' nulls with median: 30.0
Filled 'registration_date' nulls with mode: 2023-01-15 00:00:00
Filled nulls in string columns with 'Unknown'.
Removed 2 duplicate rows based on 'email'.
Standardized casing for string columns.
Corrected data types for 'user_id', 'age', and 'registration_date'.
Trimmed whitespace from string columns.

Identified 7 potential outliers in 'age' column. Reviewing...


C:\Users\sumit\AppData\Local\Temp\ipykernel_3612\3880053451.py:15: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  cleaned_df['age'] = cleaned_df['age'].fillna(median_age, inplace=True)
C:\Users\sumit\AppData\Local\Temp\ipykernel_3612\3880053451.py:22: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series thro

,name,age
1,Bob White,45
3,Charlie Green,29
4,David Blue,31
5,Eva Black,25
6,Frank Yellow,50
7,Jane Smith,24
16,Sarah Connor,35



--- Data Cleaning Complete ---

Cleaned DataFrame Head:


,user_id,name,email,age,registration_date,occupation,phone,website,company,city
0,4,Alice Brown,alice.brown@example.com,30,2022-11-01,Doctor,Unknown,unknown,Unknown,Chicago
1,5,Bob White,bob.white@example.com,45,2023-03-10,Artist,Unknown,unknown,Unknown,Houston
2,9,Glenna Reichert,chaim_mcdermott@dana.io,30,2023-01-15,Unknown,(775)976-6794 x41206,conrad.com,Yost And Sons,Bartholomebury
3,6,Charlie Green,charlie.green@example.com,29,2023-04-05,Engineer,Unknown,unknown,Unknown,Seattle
4,7,David Blue,david.blue@example.com,31,2023-05-12,Lawyer,Unknown,unknown,Unknown,Boston



Cleaned DataFrame Info:
<class 'pandas.DataFrame'>
Index: 19 entries, 0 to 20
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   user_id            19 non-null     int64         
 1   name               19 non-null     str           
 2   email              19 non-null     str           
 3   age                19 non-null     int64         
 4   registration_date  19 non-null     datetime64[us]
 5   occupation         19 non-null     str           
 6   phone              19 non-null     str           
 7   website            19 non-null     str           
 8   company            19 non-null     str           
 9   city               19 non-null     str           
dtypes: datetime64[us](1), int64(2), str(7)
memory usage: 1.6 KB


The `merged_df` has now been thoroughly cleaned, addressing nulls, duplicates, casing, types, whitespace, and identifying potential outliers. The final step in this task is to load the unified dataset to both MySQL and a CSV file, ensuring no duplicate rows are inserted into MySQL on subsequent runs.

### Task 1 (Continued): Load Unified Dataset to MySQL and CSV

Now, I will load the `cleaned_df` to two destinations:
1.  **Local CSV file**: A straightforward save to disk.
2.  **MySQL Database**: This will involve setting up a connection, creating a table, and implementing logic to prevent duplicate rows on subsequent runs. I will use the `email` column as a unique identifier for this purpose.

In [17]:
# 1. Load to CSV
output_csv_file_path = 'unified_cleaned_data.csv'
cleaned_df.to_csv(output_csv_file_path, index=False)
print(f"Unified cleaned data saved to '{output_csv_file_path}'")

# 2. Load to MySQL

import mysql.connector
import os

# --- MySQL Connection Details (UPDATE THESE WITH YOUR OWN) ---
# Replace these placeholder values with your actual MySQL database credentials
MYSQL_HOST = 'localhost'
MYSQL_USER = 'root'
MYSQL_PASSWORD = os.getenv("DB_PASSWORD")
MYSQL_DB = 'etl_database' # Database to connect to initially, or create
TABLE_NAME = 'unified_users'

try:
    # Connect to MySQL server (without specifying a database initially)
    conn = mysql.connector.connect(
        host=MYSQL_HOST,
        user=MYSQL_USER,
        password=MYSQL_PASSWORD
    )
    cursor = conn.cursor()
    print("Successfully connected to MySQL server.")

    # Create the database if it doesn't exist
    cursor.execute(f"CREATE DATABASE IF NOT EXISTS {MYSQL_DB}")
    print(f"Database '{MYSQL_DB}' ensured to exist.")

    # Now connect to the specific database
    conn.database = MYSQL_DB
    print(f"Using database '{MYSQL_DB}'.")

    # Define the table schema for MySQL, explicitly setting 'email' as a unique key
    schema_create_sql = f"""
    CREATE TABLE IF NOT EXISTS {TABLE_NAME} (
        user_id INT,
        name VARCHAR(255),
        email VARCHAR(255) PRIMARY KEY, -- Setting email as PRIMARY KEY ensures uniqueness
        age INT,
        registration_date DATE,
        occupation VARCHAR(255),
        phone VARCHAR(255),
        website VARCHAR(255),
        company VARCHAR(255),
        city VARCHAR(255)
    );
    """
    cursor.execute(schema_create_sql)
    conn.commit() # Commit changes to create table
    print(f"Table '{TABLE_NAME}' ensured to exist with 'email' as PRIMARY KEY.")

    # Load DataFrame to MySQL using INSERT IGNORE to prevent duplicates
    # This approach handles existing emails by ignoring new inserts with conflicting primary keys.

    # Convert DataFrame to list of tuples for efficient insertion
    # Ensure columns match the table schema order
    df_columns = ['user_id', 'name', 'email', 'age', 'registration_date', 'occupation', 'phone', 'website', 'company', 'city']
    # Replace NaN with None (MySQL NULL)
    cleaned_df = cleaned_df.replace({np.nan: None})
    data_to_insert = [tuple(row) for row in cleaned_df[df_columns].values]

    insert_sql = f"""
    INSERT IGNORE INTO {TABLE_NAME}
    (user_id, name, email, age, registration_date, occupation, phone, website, company, city)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
    """

    # Execute many to insert all rows
    try:
        cursor.executemany(insert_sql, data_to_insert)
        conn.commit()
        print(f"Attempted to insert {len(data_to_insert)} records into '{TABLE_NAME}'.")
        print("Existing records with duplicate emails were ignored due to PRIMARY KEY constraint.")
    except mysql.connector.Error as err:
        print(f"Error inserting data: {err}")
        conn.rollback() # Rollback in case of error

    print("Data loading to MySQL completed.")

except mysql.connector.Error as err:
    print(f"MySQL Error: {err}")
except Exception as e:
    print(f"An unexpected error occurred: {e}")
finally:
    if 'conn' in locals() and conn.is_connected():
        cursor.close()
        conn.close()
        print("MySQL connection closed.")

Unified cleaned data saved to 'unified_cleaned_data.csv'
Successfully connected to MySQL server.
Database 'etl_database' ensured to exist.
Using database 'etl_database'.
Table 'unified_users' ensured to exist with 'email' as PRIMARY KEY.
Attempted to insert 19 records into 'unified_users'.
Existing records with duplicate emails were ignored due to PRIMARY KEY constraint.
Data loading to MySQL completed.
MySQL connection closed.


After installing the libraries, please **re-run the previous code cell** (the one that starts with `# 1. Load to CSV`) to proceed with loading the data into MySQL.